In [0]:
%pip install xgboost

In [0]:
import os
import numpy as np
import pandas as pd
import joblib
import pickle

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor

In [0]:
# =========================
# Create Directories
# =========================
model_path = "/Volumes/workspace/default/ather/models/"
pred_path = "/Volumes/workspace/default/ather/predictions/"

os.makedirs(model_path, exist_ok=True)
os.makedirs(pred_path, exist_ok=True)

In [0]:
# =========================
# Load Data
# =========================
X_train = spark.read.parquet("/Volumes/workspace/default/ather/data/X_train.parquet").toPandas().values
y_train = spark.read.parquet("/Volumes/workspace/default/ather/data/y_train.parquet").toPandas().values.ravel()

X_test = spark.read.parquet("/Volumes/workspace/default/ather/data/X_test.parquet").toPandas().values
y_test = spark.read.parquet("/Volumes/workspace/default/ather/data/y_test.parquet").toPandas().values.ravel()

In [0]:
# =========================
# Initialize Models
# =========================
lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=100, random_state=42)
xgb = XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
ada = AdaBoostRegressor(n_estimators=100, random_state=42)
dt = DecisionTreeRegressor(random_state=42)

In [0]:
# =========================
# Train Models
# =========================
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)
ada.fit(X_train, y_train)
dt.fit(X_train, y_train)

In [0]:
# =========================
# Predictions
# =========================
y_pred_lr = lr.predict(X_test)
y_pred_rf = rf.predict(X_test)
y_pred_xgb = xgb.predict(X_test)
y_pred_ada = ada.predict(X_test)
y_pred_dt = dt.predict(X_test)

In [0]:
# =========================
# Save Models
# =========================
joblib.dump(lr, model_path + "linear_regression.pkl")
joblib.dump(rf, model_path + "random_forest.pkl")
joblib.dump(xgb, model_path + "xgboost.pkl")
joblib.dump(ada, model_path + "adaboost.pkl")
joblib.dump(dt, model_path + "decision_tree.pkl")

In [0]:
# =========================
# Save Predictions DataFrame
# =========================
pred_df = pd.DataFrame({
    "y_true": y_test,
    "LinearRegression": y_pred_lr,
    "RandomForest": y_pred_rf,
    "XGBoost": y_pred_xgb,
    "AdaBoost": y_pred_ada,
    "DecisionTree": y_pred_dt
})
pred_df

In [0]:
# Save as CSV
pred_df.to_csv(pred_path + "predictions.csv", index=False)

# Save as Pickle
with open(pred_path + "predictions.pkl", "wb") as f:
    pickle.dump(pred_df, f)